# Tiny GPT — pretraining on a free Colab GPU

Pretrains the **from-scratch** GPT in this repo (~57M params) on a real corpus
(TinyStories) using a free Colab **T4**.

**Important:** free Colab wipes the machine when it disconnects (idle ~90 min,
or when you close the tab). So we save checkpoints to **Google Drive** and
always resume from there — that way a disconnect just costs you the re-setup
cells, not your trained model.

First: **Runtime → Change runtime type → T4 GPU**. Then run every cell top to
bottom. After a disconnect, just run them all again — training resumes.

In [ ]:
# 1. Confirm we have a GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 2. Get the code and install deps (repo must be public, or add a token).
!git clone -b claude/gpt-model-pytorch-scratch-ckb4u7 https://github.com/mitchellmcneillharrison-png/Smallchatbot.git
%cd Smallchatbot
!pip install -q -r requirements.txt

In [ ]:
# 3. Mount Google Drive so checkpoints SURVIVE a disconnect.
#    (A popup will ask you to authorize your Google account.)
from google.colab import drive
drive.mount('/content/drive')
import os
CKPT_DIR = '/content/drive/MyDrive/tinygpt'
os.makedirs(CKPT_DIR, exist_ok=True)
print('Checkpoints will be saved to:', CKPT_DIR)

In [ ]:
# 4. Download the corpus (lives on the temp VM; re-run this after a reconnect).
!python data/get_pretrain_data.py --dataset tinystories --max_mb 200

In [ ]:
# 5. Train. Checkpoints go to Drive; --resume picks up automatically if one
#    already exists there (so this same cell works first-run AND after a
#    disconnect). The first `step 0 ...` line takes ~1 min (it runs an eval +
#    a sample first) -- don't hit stop during that quiet minute.
#
#    Want readable text sooner within one session? Use a smaller model:
#    --n_embd 512 --n_layer 6   (~19M, coherent faster).  Bigger: --n_layer 12 (~85M).
#    If you ever hit CUDA out of memory: lower --batch_size (8) or --block_size (192).
!python train.py \
  --data_path data/pretrain.txt \
  --block_size 256 --n_layer 8 --n_head 12 --n_embd 768 \
  --batch_size 16 --grad_accum 8 --amp fp16 \
  --lr 3e-4 --min_lr 3e-5 --warmup_steps 300 --max_steps 20000 \
  --eval_interval 200 --eval_iters 10 \
  --out_dir /content/drive/MyDrive/tinygpt \
  --resume /content/drive/MyDrive/tinygpt/ckpt.pt \
  --sample_prompt "Once upon a time"

In [ ]:
# 6. Generate from the trained model (run anytime a checkpoint exists in Drive).
!python generate.py --checkpoint /content/drive/MyDrive/tinygpt/ckpt.pt \
  --prompt "Once upon a time" --max_new_tokens 500 --temperature 0.8 --top_k 50

In [ ]:
# 7. Interactive: run this once, then call say("...") in any new cell.
import torch
from src.config import GPTConfig
from src.model import GPT
from src.tokenizer import CharTokenizer
from src.utils import get_device

device = get_device()
ck = torch.load('/content/drive/MyDrive/tinygpt/ckpt.pt', map_location=device)
model = GPT(GPTConfig(**ck['config'])).to(device)
model.load_state_dict(ck['model_state_dict']); model.eval()
tok = CharTokenizer(ck['vocab'])

def say(prompt, n=400, temperature=0.8, top_k=50):
    idx = torch.tensor([tok.encode(prompt)], dtype=torch.long, device=device)
    out = model.generate(idx, max_new_tokens=n, temperature=temperature, top_k=top_k)
    print(tok.decode(out[0].tolist()))

say('Once upon a time')

## Notes

- **Give the first line ~1 minute.** Step 0 runs an eval + a sample before it
  prints; that quiet minute is normal — don't press stop.
- **Disconnects are expected on free Colab** (idle ~90 min, tab closed, or ~12 h
  cap). Your checkpoint is safe in Drive. To continue: re-run cells 1–5 — cell 5
  auto-resumes from `Drive/tinygpt/ckpt.pt`. (Cells 2 and 4 re-clone and
  re-download because the VM was wiped; that's quick.)
- **Stay connected longer:** keep the tab open and in the foreground, and
  interact occasionally. For unattended / background training, Colab Pro allows it.
- **Coherence takes a while.** Loss needs to fall to ~1.2 on TinyStories; that's
  roughly 1–2 h on a T4 for the 57M model. A smaller model (see cell 5 comment)
  reaches readable text sooner if you want quicker gratification.
- This is a **text continuer**, not a Q&A bot: give it the start of a sentence
  and it continues it. (The question-answering sports bot is the separate
  `chat.py` / web demo.)